In [1]:
import pandas as pd
from urllib.parse import urlparse
import re

In [2]:
df_phishing = pd.read_csv('/Users/mmeliht/Desktop/URL-Phishing-Detection/datasets/Phishing_URL_dataset/Phishing_URLs.csv')

In [3]:
df_legitimate = pd.read_csv('/Users/mmeliht/Desktop/URL-Phishing-Detection/datasets/Phishing_URL_dataset/URL_dataset.csv')

In [4]:
####################### Extract Features and Feature Engineer ####################

In [5]:
df_phishing.head(10)

,url,Type
0,https://docs.google.com/presentation/d/e/2PACX...,Phishing
1,https://btttelecommunniccatiion.weeblysite.com/,Phishing
2,https://kq0hgp.webwave.dev/,Phishing
3,https://brittishtele1bt-69836.getresponsesite....,Phishing
4,https://bt-internet-105056.weeblysite.com/,Phishing
5,https://teleej.weebly.com/,Phishing
6,https://maryleyshon.wixsite.com/my-site-1,Phishing
7,https://chamakhman.wixsite.com/my-site-4,Phishing
8,https://posts-ch.buzz/ch/,Phishing
9,https://tinyurl.com/bdfpfyur,Phishing


In [6]:
df_legitimate.head(10)

,url,type
0,https://www.google.com,legitimate
1,https://www.youtube.com,legitimate
2,https://www.facebook.com,legitimate
3,https://www.baidu.com,legitimate
4,https://www.wikipedia.org,legitimate
5,https://www.reddit.com,legitimate
6,https://www.yahoo.com,legitimate
7,https://www.google.co.in,legitimate
8,https://www.qq.com,legitimate
9,https://www.amazon.com,legitimate


In [7]:
df_legitimate.columns

Index(['url', 'type'], dtype='object')

In [8]:
df_phishing.columns

Index(['url', 'Type'], dtype='object')

In [9]:
df_legitimate['type'].value_counts()

type
legitimate    345738
phishing      104438
Name: count, dtype: int64

In [10]:
df_phishing['Type'].value_counts()

Type
Phishing    54807
Name: count, dtype: int64

In [11]:
df_legitimate.rename(columns={"type": "Label"}, inplace=True)
df_phishing.rename(columns={"Type": "Label"}, inplace=True)

In [12]:
# Label sütununu 1 ve 0 haline getiriyoruz.
df_phishing["Label"] = df_phishing["Label"].map({
    "Phishing": 1
})

In [13]:
df_phishing['Label'].value_counts()

Label
1    54807
Name: count, dtype: int64

In [14]:
# Label sütununu 1 ve 0 haline getiriyoruz.
df_legitimate["Label"] = df_legitimate["Label"].map({
    "legitimate": 0,
    "phishing": 1
})

In [15]:
df_legitimate['Label'].value_counts()

Label
0    345738
1    104438
Name: count, dtype: int64

In [16]:
df = pd.concat([df_legitimate, df_phishing], ignore_index=True)

In [17]:
df.shape

(504983, 2)

In [18]:
df.drop_duplicates(subset="url", inplace=True)

In [19]:
df.shape

(504933, 2)

In [20]:
df['Label'].value_counts()

Label
0    345738
1    159195
Name: count, dtype: int64

In [21]:
df.isna().sum()

url      0
Label    0
dtype: int64

In [22]:
(df["url"].astype(str).str.contains(" ")).sum()

np.int64(517)

In [23]:
(df["url"].astype(str) != df["url"].astype(str).str.strip()).sum()

np.int64(1)

In [24]:
df = df[~df["url"].astype(str).str.contains(" ")]
df = df.reset_index(drop=True)

In [25]:
df.shape

(504416, 2)

In [26]:
# End Of Data Sets Concad 

In [27]:
# Extract Features Start

In [28]:
df["LengthOfURL"] = df["url"].astype(str).apply(len)

In [29]:
df.head()

,url,Label,LengthOfURL
0,https://www.google.com,0,22
1,https://www.youtube.com,0,23
2,https://www.facebook.com,0,24
3,https://www.baidu.com,0,21
4,https://www.wikipedia.org,0,25


In [30]:

#Domain içinde [.] ifadesi olduğu için domain uzunluğunu bulmada hata veriyor. Bu ifadeyi kaldırdık.

def check_bad_urls(x):
    try:
        len(urlparse(x).netloc)
        return False
    except ValueError:
        return True

df[df["url"].apply(check_bad_urls)]

,url,Label,LengthOfURL
397434,http://ladiesfirst-privileges[.]com/656465/d56...,1,54


In [31]:
# [.] ifadesini . olarak değiştirdik.
df["url"] = df["url"].astype(str).str.replace(r"\[\.\]", ".", regex=True)

In [32]:
df[df["url"].apply(check_bad_urls)]

,url,Label,LengthOfURL


In [33]:
df["DomainLengthOfURL"] = df["url"].apply(
    lambda x: len(urlparse(x).netloc)
)

In [34]:
# TLD uzunluğu bir url nin domain kısmında . dan sonraki kısıma denir .com gibi

In [35]:
df["TLDLength"] = df["url"].astype(str).apply(
    lambda x: (
        lambda parsed: len(parsed.netloc.split(":")[0].split(".")[-1])
        if "." in parsed.netloc else 0
    )(urlparse(x if "://" in x else "http://" + x))
)

In [36]:
df["IsDomainIP"] = df["url"].astype(str).apply(
    lambda x: 1 if re.fullmatch(r"\d{1,3}(\.\d{1,3}){3}", urlparse(x).netloc.split(":")[0]) else 0
)

In [37]:
# URL içinde kaç tane rakam
df["DigitCntInURL"] = df["url"].astype(str).apply(
    lambda x: sum(c.isdigit() for c in x)
)

In [38]:
#URL içinde kaç tane = karakter

In [39]:
df["EqualCharCntInURL"] = df["url"].astype(str).apply(
    lambda x: x.count("=")
)

In [40]:
#URL içinde kaç tane ? karakteri olduğu

In [41]:
df["QuesMarkCntInURL"] = df["url"].astype(str).apply(
    lambda x: x.count("?")
)

In [42]:
#Harf (a–z, A–Z), rakam (0–9) ve _ dışındaki tüm karakterler

In [43]:
df["OtherSpclCharCntInURL"] = df["url"].astype(str).apply(
    lambda x: len(re.findall(r"[^\w]", x))
)

In [44]:
#URL içindeki harf (a–z, A–Z) oranını hesaplar. URL içindeki harf (a–z, A–Z) oranını hesaplar.

In [45]:
df["URLLetterRatio"] = df["url"].astype(str).apply(
    lambda x: (sum(c.isalpha() for c in x) / len(x)) if len(x) > 0 else 0
)

In [ ]:
URL’de domain’den sonra gerçek bir path var mı?

In [ ]:
df["HavingPath"] = df["url"].astype(str).apply(
    lambda x: 1 if urlparse(x if "://" in x else "http://" + x).path not in ["", "/"] else 0
)